In [1]:
import pandas as pd
import glob
import json
import os

SAMPLES_PATH = f'{os.getcwd()}/../workspace/samples'

EXPERIMENTS = [
	('x86_64', 'full', 'x86-gemini-pro'),
	('arm64', 'full', 'arm-gemini-pro'),
	('x86_64', 'ablation', 'x86_64-krepair-ablation'),
	('arm64', 'ablation', 'arm-krepair-ablation'),
]

SCOPES = ['overall', 'include', 'exclude']

In [2]:
def parse_config(path):
	with open(path) as f:
		return {line.split('=', 1)[0] for line in f if '=' in line and not line.startswith('#')}

def normalize(option):
	return option if option.startswith('CONFIG_') else f'CONFIG_{option}'

In [8]:
def get_rows(arch, variant, folder):

	summaries = sorted(glob.glob(f'{SAMPLES_PATH}/{folder}/*/agent_repair/summary.json'))
	if not summaries:
		print(f'No samples found for {folder}')
		return []

	rows = []
	missing_configs = 0

	for path in summaries:

		repair_dir = os.path.dirname(path)
		sample = repair_dir.split('/')[-2]

		with open(path) as f:
			data = json.load(f)

		original = parse_config(f'{repair_dir}/../.config')

		for attempt in data['attempts']:

			changes = attempt['changes']
			if changes['define'] is None and changes['undefine'] is None:
				continue

			config_path = f'{repair_dir}/attempt_{attempt["attempt"]}/modified.config'
			if not os.path.exists(config_path):
				missing_configs += 1
				continue

			modified = parse_config(config_path)
			suggestions = [(o, 'include') for o in changes['define'] or []] + [(o, 'exclude') for o in changes['undefine'] or []]

			for option, action in suggestions:

				option = normalize(option)
				wants_set = action == 'include'

				rows.append({
					'arch': arch,
					'variant': variant,
					'sample': sample,
					'attempt': attempt['attempt'],
					'boot_succeeded': attempt['summary']['boot_succeeded'],
					'option': option,
					'action': action,
					'in_original': option in original,
					'in_modified': option in modified,
					'redundant': (option in original) == wants_set,
					'honored': (option in modified) == wants_set,
				})

	return rows

df = pd.DataFrame([row for experiment in EXPERIMENTS for row in get_rows(*experiment)])

scoped = pd.concat([df.assign(scope='overall'), df.assign(scope=df['action'])])
scoped['scope'] = pd.Categorical(scoped['scope'], SCOPES, ordered=True)

print(df.groupby(['arch', 'variant'])['sample'].nunique())
df.head()

arch    variant 
arm64   ablation    50
        full        50
x86_64  ablation    50
        full        50
Name: sample, dtype: int64


,arch,variant,sample,attempt,boot_succeeded,option,action,in_original,in_modified,redundant,honored
0,x86_64,full,sample_0,1,timeout,CONFIG_KASAN,exclude,True,False,False,True
1,x86_64,full,sample_0,2,panic,CONFIG_TTY,include,False,True,False,True
2,x86_64,full,sample_0,2,panic,CONFIG_SERIAL_8250,include,False,True,False,True
3,x86_64,full,sample_0,2,panic,CONFIG_SERIAL_8250_CONSOLE,include,False,True,False,True
4,x86_64,full,sample_0,2,panic,CONFIG_KASAN,exclude,True,False,False,True


In [13]:
def adherence(group):
	changes = group[~group['redundant']]
	return pd.Series({
		'suggestions': len(group),
		'honored_rate': group['honored'].mean(),
		'redundant_rate': group['redundant'].mean(),
		'non_redundant': len(changes),
		'honored_non_redundant': changes['honored'].mean(),
	})

suggestion_adherence = scoped.groupby(['scope', 'arch', 'variant'], observed=True).apply(adherence, include_groups=False)

print('Between all attempts in scope.')
for scope in SCOPES:
	print(scope)
	print(suggestion_adherence.loc[scope].to_string())
	print()

Between all attempts in scope.
overall
                 suggestions  honored_rate  redundant_rate  non_redundant  honored_non_redundant
arch   variant                                                                                  
arm64  ablation      24610.0      0.931694        0.189191        19954.0               0.915806
       full          22318.0      0.988171        0.180885        18281.0               0.985668
x86_64 ablation      23211.0      0.937487        0.221533        18069.0               0.920914
       full          18112.0      0.977363        0.182255        14811.0               0.972723

include
                 suggestions  honored_rate  redundant_rate  non_redundant  honored_non_redundant
arch   variant                                                                                  
arm64  ablation      13616.0      0.963866        0.259841        10078.0               0.951181
       full          10990.0      0.976433        0.263330         8096.0      

In [14]:
attempts = scoped.groupby(['scope', 'arch', 'variant', 'sample', 'attempt'], observed=True).agg(
	suggestions=('honored', 'size'),
	honored=('honored', 'sum'),
	boot_succeeded=('boot_succeeded', 'first'),
).reset_index()

attempts['violated'] = attempts['suggestions'] - attempts['honored']
attempts['honored_rate'] = attempts['honored'] / attempts['suggestions']
attempts['all_honored'] = attempts['violated'] == 0

attempt_adherence = attempts.groupby(['scope', 'arch', 'variant'], observed=True).agg(
	attempts=('attempt', 'size'),
	all_honored=('all_honored', 'mean'),
	mean_honored_rate=('honored_rate', 'mean'),
	mean_violated=('violated', 'mean'),
)

print('Adherence within attempts in scope.')
for scope in SCOPES:
	print(scope)
	print(attempt_adherence.loc[scope].to_string())
	print()

Adherence within attempts in scope.
overall
                 attempts  all_honored  mean_honored_rate  mean_violated
arch   variant                                                          
arm64  ablation       519     0.333333           0.919653       3.238921
       full           480     0.660417           0.990925       0.550000
x86_64 ablation       499     0.284569           0.930559       2.907816
       full           486     0.613169           0.984045       0.843621

include
                 attempts  all_honored  mean_honored_rate  mean_violated
arch   variant                                                          
arm64  ablation       485     0.550515           0.957701       1.014433
       full           458     0.644105           0.981499       0.565502
x86_64 ablation       425     0.301176           0.943335       2.218824
       full           400     0.540000           0.961933       1.002500

exclude
                 attempts  all_honored  mean_honored_rate  mea

In [19]:
paired = attempts.pivot_table(
	index=['scope', 'arch', 'sample', 'attempt'],
	columns='variant',
	values=['honored_rate', 'violated'],
	observed=True,
).dropna()

paired.columns = [f'{variant}_{metric}' for metric, variant in paired.columns]
paired = paired.reset_index()

paired['full_violated_rate'] = 1 - paired['full_honored_rate']
paired['ablation_violated_rate'] = 1 - paired['ablation_honored_rate']

for scope in SCOPES:
	print(scope)
	scope_pairs = paired[paired['scope'] == scope]
	print(scope_pairs[['full_honored_rate', 'ablation_honored_rate', 'full_violated_rate', 'ablation_violated_rate']].mean().to_string())
	print()

overall
full_honored_rate         0.989467
ablation_honored_rate     0.929093
full_violated_rate        0.010533
ablation_violated_rate    0.070907

include
full_honored_rate         0.973792
ablation_honored_rate     0.957719
full_violated_rate        0.026208
ablation_violated_rate    0.042281

exclude
full_honored_rate         0.999525
ablation_honored_rate     0.921437
full_violated_rate        0.000475
ablation_violated_rate    0.078563



In [22]:
from scipy.stats import chi2_contingency, wilcoxon

overall_suggestions = scoped[scoped['scope'] == 'overall']
overall_pairs = paired[paired['scope'] == 'overall']

per_sample = overall_suggestions.groupby(['arch', 'sample', 'variant'])['honored'].mean().unstack().dropna()

counts = overall_suggestions.groupby('variant')['honored'].agg(['sum', 'size'])
table = [[counts.loc[v, 'sum'], counts.loc[v, 'size'] - counts.loc[v, 'sum']] for v in ['full', 'ablation']]

pd.Series({
	'samples': len(per_sample),
	'full_sample_mean': per_sample['full'].mean(),
	'ablation_sample_mean': per_sample['ablation'].mean(),
	'sample_wilcoxon_p': wilcoxon(per_sample['full'], per_sample['ablation']).pvalue,
	'attempt_pairs': len(overall_pairs),
	'attempt_wilcoxon_p': wilcoxon(overall_pairs['full_honored_rate'], overall_pairs['ablation_honored_rate']).pvalue,
	'suggestion_chi2_p': chi2_contingency(table)[1],
})

samples                  1.000000e+02
full_sample_mean         9.853161e-01
ablation_sample_mean     9.366436e-01
sample_wilcoxon_p        8.576985e-13
attempt_pairs            7.870000e+02
attempt_wilcoxon_p       4.135245e-59
suggestion_chi2_p       6.477849e-277
dtype: float64

In [26]:
from functools import cache
from kconfig import Kconfig

KERNEL_SRC = f'{os.getcwd()}/../workspace/linux'
SRCARCH = {'x86_64': 'x86', 'arm64': 'arm64'}
NOT_SET = ' is not set'

kconfigs = {arch: Kconfig(KERNEL_SRC, srcarch) for arch, srcarch in SRCARCH.items()}
folders = {(arch, variant): folder for arch, variant, folder in EXPERIMENTS}

@cache
def parse_values(path):

	values = {}

	with open(path) as f:
		for line in f:
			line = line.strip()
			if line.startswith('# CONFIG_') and line.endswith(NOT_SET):
				values[line[2:-len(NOT_SET)]] = 'n'
			elif '=' in line and not line.startswith('#'):
				option, value = line.split('=', 1)
				values[option] = value

	return values

def classify(row):

	kconfig = kconfigs[row['arch']]
	name = row['option'].removeprefix('CONFIG_')
	config = parse_values(f'{SAMPLES_PATH}/{folders[(row["arch"], row["variant"])]}/{row["sample"]}/agent_repair/attempt_{row["attempt"]}/modified.config')

	if not kconfig.exists(name): return 'not in kconfig'
	if row['action'] == 'exclude' and kconfig.active_selectors(name, config): return 'selected by enabled option'
	if row['action'] == 'include' and not kconfig.dependencies_met(name, config): return 'unmet dependency'
	if not kconfig.has_prompt(name): return 'no prompt'
	return 'other'

violations = df[~df['honored']]
classified = violations.assign(reason=violations.apply(classify, axis=1))
reasons = classified.groupby(['variant', 'reason']).size().unstack('reason', fill_value=0)

print('violations by reason #')
print(reasons.to_string())

print('\nviolations by reason (%)')
print((reasons.div(reasons.sum(axis=1), axis=0) * 100).round(1).to_string())


violations by reason #
reason    no prompt  not in kconfig  other  selected by enabled option  unmet dependency
variant                                                                                 
ablation        223             834    142                        1349               584
full              0             648     12                           8                 6

violations by reason (%)
reason    no prompt  not in kconfig  other  selected by enabled option  unmet dependency
variant                                                                                 
ablation        7.1            26.6    4.5                        43.1              18.6
full            0.0            96.1    1.8                         1.2               0.9
